In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_planos"
TARGET_TABLE = "homologacao.salutar_saude.silver_planos"

DATE_COLS    = []                                            # sem colunas de data
INITCAP_COLS = ["segmentacao", "acomodacao", "coparticipacao"]  # initcap + trim: cobre variantes de case
TRIM_COLS    = ["plano_nome"]                               # trim apenas
PRICE_COLS   = ["preco_vida_mes"]                           # decimal(10,2) — formatos mistos na fonte
MERGE_KEY    = "plano_id"                                   # chave primária para o MERGE incremental

run_ts = datetime.now()

print(f"Pipeline  : silver_planos")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_planos
Iniciado  : 2026-07-04 19:09:23
Origem    : homologacao.salutar_saude.raw_planos
Destino   : homologacao.salutar_saude.silver_planos


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 28 registros lidos de homologacao.salutar_saude.raw_planos


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_initcap(col_name: str) -> F.Column:
    """Normaliza campos categóricos: remove espaços extras e padroniza capitalização.
    Não usar em siglas/códigos. Ex.: 'ENFERMARIA' → 'Enfermaria' | 'sim ' → 'Sim'.
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def parse_trim(col_name: str) -> F.Column:
    """Remove espaços extras de campos de texto livres (nome, descrição)."""
    return F.trim(F.col(col_name)).alias(col_name)


def parse_preco(col_name: str) -> F.Column:
    """Normaliza preços para decimal(10,2).
    Reutiliza a lógica de parse_valor_mensal (silver_contratos) — mesmos 3 formatos na fonte:
      - 'R$ 391,77'  → prefixo + ponto milhar + vírgula decimal
      - '472,70'     → sem prefixo, vírgula decimal
      - '680.41'     → sem prefixo, ponto decimal
    Se contém vírgula: remove 'R$ ', remove pontos (milhar), troca vírgula por ponto.
    Senão: remove 'R$ ' e usa ponto já como decimal.
    """
    c      = F.trim(F.regexp_replace(F.col(col_name), r"R\$\s*", ""))
    br_fmt = F.regexp_replace(F.regexp_replace(c, r"\.", ""), ",", ".")
    return (
        F.when(F.col(col_name).contains(","), br_fmt)
         .otherwise(c)
         .cast("decimal(10,2)")
         .alias(col_name)
    )


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)    if c in DATE_COLS    else
            parse_initcap(c) if c in INITCAP_COLS else
            parse_preco(c)   if c in PRICE_COLS   else
            parse_trim(c)    if c in TRIM_COLS    else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # em select, não withColumn
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + INITCAP_COLS + TRIM_COLS + PRICE_COLS

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
) if DQ_COLS else {}

n_silver = df_silver.count()

unexpected_acomodacao = df_silver.filter(
    F.col("acomodacao").isNotNull()
    & ~F.col("acomodacao").isin("Enfermaria", "Apartamento")
).count()

unexpected_coparticipacao = df_silver.filter(
    F.col("coparticipacao").isNotNull()
    & ~F.col("coparticipacao").isin("Sim", "Não")
).count()

# Duplicatas de chave na fonte (serão removidas no passo de escrita)
dup_keys_df  = df_silver.groupBy(MERGE_KEY).count().filter(F.col("count") > 1)
n_dup_keys   = dup_keys_df.count()
dup_ids_list = [str(r[MERGE_KEY]) for r in dup_keys_df.collect()] if n_dup_keys > 0 else []

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros                  : {n_silver:,}")
print(f"  Correspondência com RAW             : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_acomodacao else "[OK]  "
print(f"  {tag} acomodacao valores inesperados     : {unexpected_acomodacao:,}")
tag = "[WARN]" if unexpected_coparticipacao else "[OK]  "
print(f"  {tag} coparticipacao valores inesperados : {unexpected_coparticipacao:,}")
tag = "[WARN]" if n_dup_keys > 0 else "[OK]  "
detail = f" → ids: {dup_ids_list}" if dup_ids_list else ""
print(f"  {tag} {MERGE_KEY} duplicados na fonte           : {n_dup_keys:,}{detail}")
print("─" * 65)

assert n_silver == n_raw,            f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
assert unexpected_acomodacao == 0,    f"{unexpected_acomodacao} valores inesperados em 'acomodacao'"
assert unexpected_coparticipacao == 0, f"{unexpected_coparticipacao} valores inesperados em 'coparticipacao'"

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros                  : 28
  Correspondência com RAW             : [OK]
  [OK]   segmentacao: 0 nulos
  [OK]   acomodacao: 0 nulos
  [OK]   coparticipacao: 0 nulos
  [OK]   plano_nome: 0 nulos
  [OK]   preco_vida_mes: 0 nulos
  [OK]   acomodacao valores inesperados     : 0
  [OK]   coparticipacao valores inesperados : 0
  [OK]   plano_id duplicados na fonte           : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Estratégia de deduplicacão — 2 etapas:
# 1. dropDuplicates()            → remove linhas 100% idênticas
# 2. dropDuplicates([MERGE_KEY]) → garante cardinalidade 1:1 por chave para o MERGE Delta
df_to_merge = df_silver.dropDuplicates().dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza plano existente se houver mudança
        .whenNotMatchedInsertAll()    # insere novo plano
        # .whenNotMatchedBySourceDelete()  # descomente para remover planos deletados da RAW
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_to_merge.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_planos
[OK] Registros na tabela  : 28
[OK] Fim do pipeline      : 2026-07-04 19:09:44


In [0]:
%sql
SELECT * FROM homologacao.salutar_saude.silver_planos
ORDER BY plano_id

plano_id,operadora_id,plano_nome,segmentacao,acomodacao,coparticipacao,preco_vida_mes,_updated_at
1,1,Plano Clássico 242,Ambulatorial,Enfermaria,Não,391.77,2026-07-04T19:09:23.697Z
2,1,Plano Clássico 338,Ambulatorial,Apartamento,Sim,202.94,2026-07-04T19:09:23.697Z
3,1,Plano Flex 529,Ambulatorial,Enfermaria,Não,680.41,2026-07-04T19:09:23.697Z
4,1,Plano Premium 384,Ambulatorial + Hospitalar,Enfermaria,Sim,717.57,2026-07-04T19:09:23.697Z
5,1,Plano Premium 452,Ambulatorial + Hospitalar,Enfermaria,Sim,472.54,2026-07-04T19:09:23.697Z
6,2,Plano Premium 743,Hospitalar + Obstetrícia,Enfermaria,Sim,240.68,2026-07-04T19:09:23.697Z
7,2,Plano Top 333,Ambulatorial + Hospitalar,Enfermaria,Não,233.56,2026-07-04T19:09:23.697Z
8,2,Plano Master 750,Ambulatorial,Enfermaria,Sim,472.70,2026-07-04T19:09:23.697Z
9,3,Plano Top 799,Ambulatorial + Hospitalar,Apartamento,Sim,696.03,2026-07-04T19:09:23.697Z
10,3,Plano Premium 755,Hospitalar,Enfermaria,Sim,535.94,2026-07-04T19:09:23.697Z
